# 029 · Weight Initialization, Part 1 — What NOT To Do

Four ways to initialise a network badly. Each fails differently, and the
difference is the whole point — two of them fail *invisibly*.

| Part | What we reproduce |
|---|---|
| A | all zeros → gradients are exactly 0, training never starts |
| B | all the same constant → every neuron stays identical |
| C | too small (×0.01) → activation std **0.156 → 0.000 by layer 6** |
| D | too large (×1.0) → std holds at **0.974** and looks fine, but **87% saturated** |

Needs `numpy` only. No training, no GPU — the damage is done before the first
gradient is computed.

In [ ]:
import numpy as np

N, WIDTH, DEPTH = 512, 256, 12

tanh = np.tanh
def relu(z):
    return np.maximum(0.0, z)

## Part A — All zeros

The tempting choice, and the worst one. `tanh(0) = 0` and `relu(0) = 0`, so every
activation is zero, and the gradient of the loss with respect to every weight
picks up that zero as a factor.

In [ ]:
# One layer, by hand, so nothing is hidden.
x = np.random.default_rng(0).standard_normal((8, 5))
W = np.zeros((5, 4))

z = x @ W
a = np.tanh(z)
print("pre-activation z :", np.abs(z).max())
print("activation   a   :", np.abs(a).max())

# dL/dW = x.T @ (upstream * tanh'(z)) and tanh'(0) = 1, so the zero that
# kills it comes from a, not from the derivative. Follow it one layer on:
upstream = np.ones((8, 4))
dW_next = a.T @ upstream          # gradient for the NEXT layer's weights
print("gradient for the next layer's W :", np.abs(dW_next).max())

assert np.abs(a).max() == 0.0
assert np.abs(dW_next).max() == 0.0

In [ ]:
# Stack it and the whole network is inert.
a = np.random.default_rng(0).standard_normal((N, WIDTH))
for layer in range(DEPTH):
    W = np.zeros((a.shape[1], WIDTH))
    a = np.tanh(a @ W)
print("activation std after", DEPTH, "layers:", a.std())
print("There is no signal to propagate and no gradient to descend.")
assert a.std() == 0.0

## Part B — All the same non-zero constant

Gradients are fine this time. The failure is subtler: every neuron in a layer
sees the same inputs and holds the same weights, so it computes the same thing —
and receives the same gradient, so it *stays* identical forever.

In [ ]:
rng = np.random.default_rng(1)
x = rng.standard_normal((16, 6))
W = np.full((6, 4), 0.3)          # four neurons, all identical
b = np.zeros(4)

a = np.tanh(x @ W + b)
print("the four neurons' outputs, first 3 rows:")
print(np.round(a[:3], 6))

# Every column is the same column.
assert np.allclose(a[:, 0], a[:, 1]) and np.allclose(a[:, 0], a[:, 3])
print("\nAll four columns identical:", np.allclose(a, a[:, [0]]))
print("A layer of 4 neurons is doing the work of 1.")

In [ ]:
# And training does not break it. A proper two-layer network, every hidden
# unit initialised the same, trained by hand for 200 steps.
rng = np.random.default_rng(7)
X = rng.standard_normal((32, 6))
y = rng.standard_normal((32, 1))

W1 = np.full((6, 4), 0.3)          # 4 hidden units, all identical
W2 = np.full((4, 1), 0.3)          # and identical outgoing weights

for step in range(200):
    h = np.tanh(X @ W1)
    out = h @ W2
    d_out = 2 * (out - y) / len(X)
    d_h = (d_out @ W2.T) * (1 - h ** 2)
    W2 = W2 - 0.05 * (h.T @ d_out)
    W1 = W1 - 0.05 * (X.T @ d_h)

print("incoming weights of the 4 hidden units still identical:",
      np.allclose(W1, W1[:, [0]]))
print("outgoing weights still identical:", np.allclose(W2, W2[0]))
print("\nEach hidden unit sees the same inputs, holds the same weights, so it")
print("computes the same value and receives the same gradient. It can never")
print("differentiate itself. Randomness exists to break SYMMETRY, not for luck.")
assert np.allclose(W1, W1[:, [0]])
assert np.allclose(W2, W2[0])

## Parts C and D — Random, but the wrong size

Now the interesting failures. Random weights break symmetry, so both of these
*train*. They just train badly, and one of them looks perfectly healthy while
doing it.

Propagate a signal through 12 layers of width 256 and watch the activation
standard deviation.

In [ ]:
def propagate(scale, act, seed=0):
    """Return (std per layer, fraction saturated per layer)."""
    rng = np.random.default_rng(seed)
    a = rng.standard_normal((N, WIDTH))
    stds, sat = [], []
    for _ in range(DEPTH):
        W = rng.standard_normal((a.shape[1], WIDTH)) * scale(a.shape[1])
        a = act(a @ W)
        stds.append(float(a.std()))
        sat.append(float((np.abs(a) > 0.99).mean()))
    return stds, sat


SCHEMES = [
    ("small x0.01  (tanh)", lambda fan: 0.01, tanh),
    ("large x1.0   (tanh)", lambda fan: 1.0, tanh),
    ("small x0.01  (ReLU)", lambda fan: 0.01, relu),
]

print(f"{'scheme':<24}" + "".join(f"L{i+1:<7}" for i in (0, 2, 5, 8, 11)))
for name, scale, act in SCHEMES:
    stds, _ = propagate(scale, act)
    print(f"{name:<24}" + "".join(f"{stds[i]:<8.3f}" for i in (0, 2, 5, 8, 11)))

In [ ]:
small, _ = propagate(lambda fan: 0.01, tanh)
print("too small, layer by layer:")
for i, s in enumerate(small[:7], 1):
    print(f"  layer {i:>2}: std = {s:.6f}")
print("\nThe signal is gone by layer 6. A network cannot learn from nothing.")
assert small[0] < 0.2 and small[5] < 1e-3

### The one that looks healthy

Now the large case. The standard deviation holds steady near **0.974** all the
way down — which is exactly what you would want to see, and exactly why this
failure is dangerous.

In [ ]:
large, large_sat = propagate(lambda fan: 1.0, tanh)
print("std:", [round(s, 3) for s in large[:6]], "...")
print("\nLooks perfect. Now look at WHERE those activations are:")
print(f"  fraction with |a| > 0.99, layer 1  : {large_sat[0]:.0%}")
print(f"  fraction with |a| > 0.99, layer 12 : {large_sat[-1]:.0%}")

assert large[-1] > 0.9          # std looks fine
assert large_sat[-1] > 0.85     # but it is saturation, not signal

In [ ]:
# tanh is flat out there, so the gradient that has to flow back is tiny.
for a_val in (0.0, 0.5, 0.9, 0.99):
    print(f"  tanh output {a_val:<5} -> derivative 1 - a^2 = {1 - a_val**2:.4f}")

print("\nStd of 0.974 with 87% saturated means the layer is a wall of +/-1.")
print("Backprop multiplies by ~0.02 at every layer. 0.02^12 is about 4e-21.")
print(f"\n0.02 ** 12 = {0.02 ** 12:.2e}")
assert abs((1 - 0.99 ** 2) - 0.0199) < 1e-6

### The same mistake with ReLU looks completely different

ReLU has no ceiling, so too-large weights do not saturate — they **explode**.
Same cause, opposite symptom.

In [ ]:
rng = np.random.default_rng(0)
a = rng.standard_normal((N, WIDTH))
for layer in range(1, DEPTH + 1):
    W = rng.standard_normal((a.shape[1], WIDTH)) * 1.0
    a = relu(a @ W)
    if layer in (1, 2, 3, 4, 5):
        print(f"  layer {layer}: std = {a.std():.3e}")

print("\nSaturation with tanh, explosion with ReLU. One diagnosis: the")
print("spread of the weights has to depend on the ARCHITECTURE.")

## What to take away

- **All zeros:** `g(0) = 0`, so activations and gradients are exactly 0 and
  training never starts.
- **All one constant:** gradients are fine, but every neuron stays identical —
  the layer collapses to one neuron. **Randomness exists to break symmetry.**
- **Too small (×0.01):** activation std **0.156 → 0.000 by layer 6**. Signal gone.
- **Too large (×1.0):** std holds at **0.974** and *looks healthy* — but **87%
  of activations are saturated**, where tanh's derivative is **0.0199**.
- A neuron with 250 inputs is not a neuron with 5. **The right spread depends on
  the fan-in** — which is lesson 030.

## Exercises

1. In Part D, find by bisection the scale at which tanh saturation first exceeds
   50% at layer 12. How close is it to `1/sqrt(256)`?
2. The zero-init argument used `tanh`. Does it still hold for a **sigmoid**
   network, where `sigmoid(0) = 0.5`? Work out which part of the argument
   survives and which does not.
3. Part B broke symmetry with random weights. Try random **biases** with zero
   weights instead. Does that break symmetry? Explain the result.
4. Build a 12-layer tanh network in Keras with `kernel_initializer='zeros'`,
   train for one epoch on any data, and confirm the weights are bit-identical
   afterwards.
5. Reduce `WIDTH` from 256 to 16 and rerun Part C. At which layer does the signal
   die now? What does that tell you about the relationship you need?